# Exp 08: Best model Macro F1

Card: [exp08_best_model_f1.md](../../experiments/exp08_best_model_f1.md)


In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_recall_curve

from core.config import CLASSES, ROOT
from core.cv import CVResult, Config
from core.scores import Scores

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    SAVE_FIGURES = False
else:
    SAVE_FIGURES = True
CONFIG = Config.from_toml(
    ROOT / "sections" / "model" / "configs" / "exp04_oversampling_vip" / "resnet18.toml"
)
FIGURES = ROOT / "sections" / "model" / "figures"
PREFIX = "exp08_best_model_f1"

# dataviz reference palette, light mode; exp04 keeps its color from exp07
COLOR = "#eb6834"
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#52514e",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.titlesize": "medium",
        "axes.titlecolor": "#52514e",
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "legend.frameon": False,
        "legend.fontsize": "small",
    }
)

result = CVResult.load(CONFIG)
oof = result.oof

## Macro F1


In [ ]:
# tuned on same rows: thresholds fit and scored on all out-of-fold rows (optimistic)
same = result.scores.best_f1()


def rows(fold):
    part = oof[oof["fold"] == fold]
    return part[CLASSES].to_numpy(), part[[f"p_{c}" for c in CLASSES]].to_numpy()


# tuned on other fold: thresholds of one fold scored on the other
cross = {}
for fold in CONFIG.folds:
    (other,) = [f for f in CONFIG.folds if f != fold]
    thresholds = Scores(*rows(other)).best_f1()["threshold"].to_numpy()
    y, p = rows(fold)
    cross[f"F1 fold {fold}"] = f1_score(
        y, p >= thresholds, average=None, zero_division=0
    )
cross = pd.DataFrame(cross, index=CLASSES)
cross["F1 mean"] = cross.mean(axis=1)

table = pd.concat({"tuned on same rows": same, "tuned on other fold": cross}, axis=1)
table.loc["Macro F1"] = table.mean()
table.loc["Macro F1", ("tuned on same rows", ["threshold", "precision", "recall"])] = (
    np.nan
)
table.round(3)

## Precision-recall curves


In [ ]:
fig, axes = plt.subplots(
    1, len(CLASSES), figsize=(3.2 * len(CLASSES), 3.4), sharey=True
)
for cls, ax in zip(CLASSES, axes):
    precision, recall, _ = precision_recall_curve(oof[cls], oof[f"p_{cls}"])
    ax.plot(recall, precision, color=COLOR, lw=2, drawstyle="steps-post")
    best = same.loc[cls]
    ax.plot(
        best["recall"], best["precision"], "o", ms=8, color=COLOR, mec="#fcfcfb", mew=2
    )
    ax.annotate(
        f"F1 {best['f1']:.3f}\nt {best['threshold']:.3f}",
        (best["recall"], best["precision"]),
        xytext=(-8, -10),
        textcoords="offset points",
        ha="right",
        va="top",
        fontsize=8,
        color="#52514e",
    )
    ax.set(
        title=f"{cls} ({int(oof[cls].sum())} positives)",
        xlabel="recall",
        xlim=(0, 1.02),
        ylim=(0, 1.05),
    )
axes[0].set_ylabel("precision")
fig.suptitle(
    "exp04 ResNet-18, out-of-fold rows of folds 0 and 1; dot = best-F1 threshold",
    color="#52514e",
)
fig.tight_layout()
figures = {f"{PREFIX}_pr_curves.png": fig}
plt.show()

In [ ]:
if SAVE_FIGURES:
    for fname, fig in figures.items():
        fig.savefig(FIGURES / fname, dpi=120)